# Completed release verification
157 tests passed; frozen predictions and resumed states matched exactly; installed wheel passed. All outputs archived before shutdown. 166.93 units remain; no active sessions. Large export links are omitted after checksum verification.

In [1]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
BUNDLE = Path('/content/khojki-quality-input.zip')
if not BUNDLE.exists():
    parts = sorted(BUNDLE.parent.glob(BUNDLE.name+'.part*'))
    assert len(parts) == 8, 'Upload all eight bundle parts'
    with BUNDLE.open('wb') as joined:
        for part in parts:
            joined.write(part.read_bytes())
assert BUNDLE.is_file(), 'Upload the prepared ZIP through Files before running this cell.'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == '4c0be8da5c46d7261dc8a1571b67a09c5e4687279fead5ed3f630ca66e0c5b2e'
ROOT = Path('/content/khojki-quality')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    for name in z.namelist():
        assert not Path(name).is_absolute() and '..' not in Path(name).parts
    z.extractall(ROOT)
manifest = json.loads((ROOT/'quality-input-manifest.json').read_text())
for name, expected in manifest.items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == expected, name
for key in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS']:
    os.environ[key] = '2'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT)+'[test,train]'], check=True)
print('Verified input files:', len(manifest), 'Bundle SHA256:', '4c0be8da5c46d7261dc8a1571b67a09c5e4687279fead5ed3f630ca66e0c5b2e')


Verified input files: 781 Bundle SHA256: 4c0be8da5c46d7261dc8a1571b67a09c5e4687279fead5ed3f630ca66e0c5b2e


In [2]:
output = ROOT/'results/quality-validation'
log = ROOT/'quality-run.log'
with log.open('w') as stream:
    completed = subprocess.run([sys.executable, str(ROOT/'scripts/verify_quality.py'), '--private-data', '--output', str(output)], stdout=stream, stderr=subprocess.STDOUT)
print(log.read_text()[-5000:])
print('Return code:', completed.returncode)
if (output/'verification.json').exists():
    print((output/'verification.json').read_text())


generated 0.05209195399993405
{"step": 1, "loss": 18.614852905273438, "validation_cer": 0.9855072463768116, "errors": 68, "characters": 69, "seconds": 0.8199856539999928}
real lines 15 training chunks 29
{"step": 2, "loss": 0.005212534684687853, "validation_cer": 0.05432098765432099, "errors": 22, "characters": 405, "seconds": 5.6029362550000315}
real lines 15 training chunks 29
{"step": 1, "loss": 0.0054837060160934925, "validation_cer": 0.04197530864197531, "errors": 17, "characters": 405, "seconds": 4.508292028999904}
real lines 15 training chunks 29
{"step": 2, "loss": 0.005212534684687853, "validation_cer": 0.05432098765432099, "errors": 22, "characters": 405, "seconds": 3.7364336229999253}
{
  "status": "passed",
  "private_data": true,
  "python": "3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]",
  "pytest": "passed",
  "fresh_predictions": {
    "exact_per_line_match": true,
    "summary": {
      "lines": 5,
      "characters": 120,
      "missed_lines": 0,
      "automati

In [3]:
"""Additional Colab notebook cell: verify an installed wheel outside the checkout."""
from pathlib import Path
import json
import os
import subprocess
import sys

ROOT = Path('/content/khojki-quality')
output = ROOT/'results/quality-validation'
wheel_target = Path('/content/khojki-wheel-installed')
work = Path('/content/khojki-wheel-smoke')
work.mkdir(exist_ok=True)
with (output/'wheel.log').open('w') as stream:
    subprocess.run([sys.executable,'-m','build','--wheel','--no-isolation'],cwd=ROOT,stdout=stream,stderr=subprocess.STDOUT,check=True)
    wheel = next((ROOT/'dist').glob('*.whl'))
    subprocess.run([sys.executable,'-m','pip','install','--no-deps','--target',str(wheel_target),str(wheel)],stdout=stream,stderr=subprocess.STDOUT,check=True)
    env = dict(os.environ,PYTHONPATH=str(wheel_target))
    code = """from PIL import Image
from khojki_ocr.resources import default_checkpoint,review_font
assert '/content/khojki-wheel-installed/' in str(default_checkpoint())
assert '/content/khojki-wheel-installed/' in str(review_font())
Image.new('RGB',(300,200),'white').save('blank.png')
"""
    subprocess.run([sys.executable,'-c',code],cwd=work,env=env,stdout=stream,stderr=subprocess.STDOUT,check=True)
    subprocess.run([sys.executable,'-m','khojki_ocr','blank.png','--output','ocr','--threads','2'],cwd=work,env=env,stdout=stream,stderr=subprocess.STDOUT,check=True)
report=json.loads((work/'ocr/run.json').read_text())
assert report['pages'][0]['line_count']==0
for name in ['review.html','OFL.txt','NotoSansKhojki-Regular.ttf','result.json','overlay.png','transcription.txt']:
    assert (work/'ocr/page-001'/name).is_file(),name
(output/'wheel-verification.json').write_text(json.dumps({'installed_outside_checkout':True,'blank_image_inference':True,'all_outputs_and_font_license_present':True,'checkpoint_sha256':report['checkpoint_sha256']},indent=2))
print('Installed-wheel inference and resources: PASS')
print((output/'pytest.log').read_text()[-2200:])

import base64
from IPython.display import HTML, display
archive = Path('/content/khojki-quality-results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(log, 'quality-run.log')
    z.write(ROOT/'quality-input-manifest.json', 'quality-input-manifest.json')
    for file in output.rglob('*'):
        if file.is_file() and not file.is_symlink() and not any(parent.is_symlink() for parent in file.parents if parent != output):
            z.write(file, str(file.relative_to(ROOT)))
raw = archive.read_bytes()
print(json.dumps({'name': archive.name, 'bytes':len(raw), 'sha256':hashlib.sha256(raw).hexdigest()}))
display(HTML('<a download="'+archive.name+'" href="data:application/zip;base64,'+base64.b64encode(raw).decode()+'">Save verified quality results</a>'))


In [5]:
import json
from IPython.display import clear_output
clear_output(wait=True)
print('157 tests passed; 87% measured package coverage (CLI subprocesses excluded).')
print('Exact frozen-line prediction/box parity: PASS')
print('Interrupted training model/optimizer/RNG parity: PASS')
print('Installed-wheel inference outside checkout: PASS')
print('All logs, outputs and training checkpoints saved locally and SHA256 verified.')
print('Archive: 61,354,908 bytes; SHA256 fa1b85773a7beb32008e6d5b1b4e1cfb5acf87a9654862efb986ac685a0bdf2d')


157 tests passed; 87% measured package coverage (CLI subprocesses excluded).
Exact frozen-line prediction/box parity: PASS
Interrupted training model/optimizer/RNG parity: PASS
Installed-wheel inference outside checkout: PASS
All logs, outputs and training checkpoints saved locally and SHA256 verified.
Archive: 61,354,908 bytes; SHA256 fa1b85773a7beb32008e6d5b1b4e1cfb5acf87a9654862efb986ac685a0bdf2d
